In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'finder_core').is_dir())))
"""Leakage-controlled GJR-GARCH(1,1)-skewed-t benchmark for crash/anomaly labels.

Expected input layout (same convention as 05 MultiBenchmarks.ipynb):
    Data/Indicators_crash/{MARKET}_df_Indicators.csv
    Data/Indicators_crash/{MARKET}_df_Crash_Label.csv

The model is estimated on close-to-close log returns only.  At target date t,
all estimation/filtering information ends at t-1.  The model's one-step-ahead
skewed-t conditional distribution is used to compute

    P(r_t <= q_t | F_{t-1}),

where q_t is the exact rolling crash threshold from the label file (preferred),
or an explicitly configured rolling quantile if that threshold is unavailable.

Important: this script does NOT infer the event definition from the labels.  Set
EVENT_THRESHOLD_COLUMN or the RECOMPUTE_THRESHOLD settings to match the label
construction used in the manuscript.  The script compares the threshold-based
reconstructed label with crash_label and reports any disagreement.

Install dependencies:
    pip install numpy pandas scikit-learn arch matplotlib

Run from the repository/project root:
    python GJR_GARCH_Benchmark.py
"""

from __future__ import annotations

import warnings
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

try:
    from arch import arch_model
    from arch.univariate import SkewStudent
except ImportError as exc:  # pragma: no cover - message is for the user
    raise ImportError(
        "The 'arch' package is required. Install it with: pip install arch"
    ) from exc

In [ ]:
# =============================================================================
# 1. USER CONFIGURATION
# =============================================================================

MARKET = "FTSE"  # e.g. SP500, STOXX, CAC, DAX, FTSE, CBU0
DATA_DIR = Path("Data")
INDICATORS_DIR = DATA_DIR / "Indicators_crash"

INDICATOR_FILE = INDICATORS_DIR / f"{MARKET}_df_Indicators.csv"
LABEL_FILE = INDICATORS_DIR / f"{MARKET}_df_Crash_Label.csv"

# Must match the final test cutoff used in the ML benchmark notebook.
TEST_CUTOFF = pd.Timestamp("2025-03-01")

# Return column preference. If none is found, log returns are computed from close.
RETURN_COLUMN = "logreturn"
CLOSE_COLUMN = "close"
RETURNS_ARE_PERCENT = False  # False when logreturn is stored as a decimal, e.g. 0.01
THRESHOLDS_ARE_PERCENT = False  # Usually False; match the units in the crash-label CSV.

# Use the exact target threshold from the label-construction file wherever possible.
# Your existing notebook explicitly excludes `crash_threshold` from model features,
# so this is the first column to check.
EVENT_THRESHOLD_COLUMN = "crash_threshold"

# Only used when EVENT_THRESHOLD_COLUMN is absent and RECOMPUTE_THRESHOLD=True.
# Set these to the EXACT values used to construct crash_label; do not assume 5%/126
# unless that is actually your label definition.
RECOMPUTE_THRESHOLD = False
ROLLING_TAIL_PROBABILITY = 0.05
ROLLING_LOOKBACK = 252
ROLLING_MIN_PERIODS = 252

# Estimation / forecast protocol.
MIN_TRAIN_OBSERVATIONS = 252
REFIT_EVERY = 21  # expanding-window re-estimation every 21 target observations
                   # Set to 1 for a fully expanding daily refit sensitivity test.
MAX_OPTIMIZER_ITERATIONS = 3000

# Labels / decision thresholds.
FIXED_PROBABILITY_THRESHOLD = 0.50
INITIAL_SEQUENTIAL_THRESHOLD = 0.50

# Moving-block bootstrap for uncertainty intervals (test-set forecast/label pairs).
BOOTSTRAP_REPLICATES = 1000
BOOTSTRAP_BLOCK_SIZE = 20
BOOTSTRAP_RANDOM_STATE = 42

# Optional paired comparison against the existing ML test-predictions CSV.
# Example:
# COMPARATOR_PREDICTIONS_CSV = Path(
#     "Results/Benchmark_Outputs/Without_news_v2/CBU0_finderBenchmark_test_predictions.csv"
# )
COMPARATOR_PREDICTIONS_CSV: Optional[Path] = None
COMPARATOR_PROBABILITY_COLUMNS: Sequence[str] = ("Logistic_probability", "LSTM_probability")

# If True, stop when crash_label != (logreturn <= threshold) for any comparable row.
# Initially False so the script can report the mismatches for diagnosis.
STRICT_LABEL_THRESHOLD_MATCH = False

SAVE_RESULTS = True
OUTPUT_DIR = Path("Results/Benchmark_Outputs/GJR_GARCH")
RANDOM_SEED = 42

In [ ]:
# =============================================================================
# 2. DATA PREPARATION
# =============================================================================


def find_column(df: pd.DataFrame, requested: str) -> Optional[str]:
    """Find a column without case sensitivity; return the actual column name."""
    lookup = {str(col).strip().lower(): col for col in df.columns}
    return lookup.get(requested.strip().lower())


def load_and_align_data() -> Tuple[pd.DataFrame, str, str]:
    """Load indicators and crash labels and align them by trading date."""
    if not INDICATOR_FILE.exists():
        raise FileNotFoundError(f"Indicator file not found: {INDICATOR_FILE.resolve()}")
    if not LABEL_FILE.exists():
        raise FileNotFoundError(f"Crash-label file not found: {LABEL_FILE.resolve()}")

    indicators = pd.read_csv(INDICATOR_FILE)
    labels = pd.read_csv(LABEL_FILE)

    indicator_date_col = find_column(indicators, "Date")
    label_date_col = find_column(labels, "date") or find_column(labels, "Date")
    label_col = find_column(labels, "crash_label")

    if indicator_date_col is None:
        raise KeyError(f"Expected a Date column in {INDICATOR_FILE}")
    if label_date_col is None:
        raise KeyError(f"Expected a date/Date column in {LABEL_FILE}")
    if label_col is None:
        raise KeyError(f"Expected a crash_label column in {LABEL_FILE}")

    indicators = indicators.copy()
    labels = labels.copy()

    # Preserve the event threshold from the crash-label CSV explicitly. If the
    # indicators table also contains a same-named rolling feature, it must not
    # accidentally shadow the threshold that generated crash_label.
    label_threshold_col = (
        find_column(labels, EVENT_THRESHOLD_COLUMN) if EVENT_THRESHOLD_COLUMN else None
    )
    if label_threshold_col is not None:
        labels["_event_threshold_from_label"] = pd.to_numeric(
            labels[label_threshold_col], errors="coerce"
        )

    indicators[indicator_date_col] = pd.to_datetime(
        indicators[indicator_date_col], errors="coerce"
    )
    labels[label_date_col] = pd.to_datetime(labels[label_date_col], errors="coerce")
    labels[label_col] = pd.to_numeric(labels[label_col], errors="coerce")

    indicators = (
        indicators.dropna(subset=[indicator_date_col])
        .sort_values(indicator_date_col)
        .drop_duplicates(subset=[indicator_date_col], keep="last")
    )
    labels = (
        labels.dropna(subset=[label_date_col, label_col])
        .sort_values(label_date_col)
        .drop_duplicates(subset=[label_date_col], keep="last")
    )

    indicators = indicators.rename(columns={indicator_date_col: "date"})
    labels = labels.rename(columns={label_date_col: "date", label_col: "crash_label"})

    merged = indicators.merge(
        labels,
        on="date",
        how="inner",
        validate="one_to_one",
        suffixes=("", "_label"),
    )
    merged = (
        merged.sort_values("date")
        .drop_duplicates(subset=["date"], keep="last")
        .reset_index(drop=True)
    )

    if merged.empty:
        raise ValueError("No rows remain after aligning indicator and crash-label dates.")

    merged["crash_label"] = pd.to_numeric(merged["crash_label"], errors="coerce")
    merged = merged.dropna(subset=["crash_label"]).copy()
    merged["crash_label"] = merged["crash_label"].astype(int)

    if not set(merged["crash_label"].unique()).issubset({0, 1}):
        raise ValueError("crash_label must contain only binary values 0 and 1.")

    # Return series: prefer the supplied logreturn column to maintain consistency
    # with the target definition. If unavailable, compute log(C_t/C_{t-1}).
    return_col = find_column(merged, RETURN_COLUMN)
    if return_col is None:
        return_col = find_column(merged, "log_return")
    if return_col is None:
        return_col = find_column(merged, "log_return_1d")

    if return_col is not None:
        merged["_log_return_decimal"] = pd.to_numeric(merged[return_col], errors="coerce")
        return_source = return_col
        if RETURNS_ARE_PERCENT:
            merged["_log_return_decimal"] = merged["_log_return_decimal"] / 100.0
    else:
        close_col = find_column(merged, CLOSE_COLUMN)
        if close_col is None:
            raise KeyError(
                f"No {RETURN_COLUMN}/log_return column or {CLOSE_COLUMN} column was found."
            )
        close = pd.to_numeric(merged[close_col], errors="coerce")
        close = close.where(close > 0)
        merged["_log_return_decimal"] = np.log(close).diff()
        return_source = f"computed log difference of {close_col}"

    # The fitted ARCH model is scaled to percentage points for numerical stability.
    merged["_log_return_decimal"] = pd.to_numeric(
        merged["_log_return_decimal"], errors="coerce"
    )
    merged.loc[~np.isfinite(merged["_log_return_decimal"]), "_log_return_decimal"] = np.nan

    # Find the event threshold. Prefer the exact threshold stored with crash labels.
    threshold_col = None
    if "_event_threshold_from_label" in merged.columns:
        threshold_col = "_event_threshold_from_label"
        merged["event_threshold_decimal"] = pd.to_numeric(
            merged[threshold_col], errors="coerce"
        )
        threshold_source = f"{threshold_col} (from label CSV: {EVENT_THRESHOLD_COLUMN})"
    else:
        threshold_col = find_column(merged, EVENT_THRESHOLD_COLUMN) if EVENT_THRESHOLD_COLUMN else None

    if threshold_col is not None and "event_threshold_decimal" not in merged.columns:
        merged["event_threshold_decimal"] = pd.to_numeric(
            merged[threshold_col], errors="coerce"
        )
        threshold_source = threshold_col
    elif threshold_col is None and RECOMPUTE_THRESHOLD:
        # The shift(1) is essential: q_t is calculated using observations through t-1.
        merged["event_threshold_decimal"] = (
            merged["_log_return_decimal"]
            .shift(1)
            .rolling(
                window=ROLLING_LOOKBACK,
                min_periods=ROLLING_MIN_PERIODS,
            )
            .quantile(ROLLING_TAIL_PROBABILITY)
        )
        threshold_source = (
            f"shifted rolling q={ROLLING_TAIL_PROBABILITY:.3f}, "
            f"lookback={ROLLING_LOOKBACK}"
        )
    elif threshold_col is None:
        raise KeyError(
            f"Threshold column '{EVENT_THRESHOLD_COLUMN}' was not found. To preserve the "
            "paper's event definition, either set EVENT_THRESHOLD_COLUMN to the actual "
            "threshold column in the label CSV, or set RECOMPUTE_THRESHOLD=True and "
            "configure ROLLING_TAIL_PROBABILITY / ROLLING_LOOKBACK / ROLLING_MIN_PERIODS "
            "to exactly match the label-generation code."
        )

    if THRESHOLDS_ARE_PERCENT:
        merged["event_threshold_decimal"] = merged["event_threshold_decimal"] / 100.0
    merged.loc[
        ~np.isfinite(merged["event_threshold_decimal"]), "event_threshold_decimal"
    ] = np.nan

    print("DATA ALIGNMENT")
    print("=" * 72)
    print(f"Market:             {MARKET}")
    print(f"Rows:               {len(merged)}")
    print(f"Date range:         {merged['date'].min().date()} -> {merged['date'].max().date()}")
    print(f"Return source:      {return_source}")
    print(f"Threshold source:   {threshold_source}")
    print(f"Label prevalence:   {merged['crash_label'].mean():.4f}")
    print(f"Label counts:       {merged['crash_label'].value_counts().sort_index().to_dict()}")

    # Diagnostics: verify that the event threshold and supplied binary label agree.
    valid_check = (
        merged["_log_return_decimal"].notna()
        & merged["event_threshold_decimal"].notna()
    )
    if valid_check.any():
        threshold_label = (
            merged.loc[valid_check, "_log_return_decimal"]
            <= merged.loc[valid_check, "event_threshold_decimal"]
        ).astype(int)
        given_label = merged.loc[valid_check, "crash_label"].astype(int)
        mismatch_count = int((threshold_label.to_numpy() != given_label.to_numpy()).sum())
        mismatch_rate = mismatch_count / int(valid_check.sum())
        print(
            f"Threshold/label consistency: {1.0 - mismatch_rate:.4%} "
            f"({mismatch_count:,} mismatches across {int(valid_check.sum()):,} rows)"
        )
        if mismatch_count and STRICT_LABEL_THRESHOLD_MATCH:
            raise ValueError(
                "crash_label does not match (logreturn <= event_threshold). "
                "Check target horizon, threshold timing/units, and return definition."
            )
        if mismatch_count:
            warnings.warn(
                "Some crash labels do not equal (logreturn <= event_threshold). "
                "The script will evaluate the supplied crash_label but compute the "
                "GARCH probability against event_threshold. Resolve this discrepancy "
                "before using the results in the manuscript."
            )

    return merged, return_source, threshold_source

In [ ]:
# =============================================================================
# 3. GJR-GARCH(1,1) WITH HANSEN SKEWED-STUDENT-t INNOVATIONS
# =============================================================================


def fit_gjr_skewt(returns_pct: pd.Series) -> Dict[str, float]:
    """Fit GJR-GARCH(1,1)-skewt to returns known at the forecast origin.

    Specification:
        r_t = mu + epsilon_t, epsilon_t = sigma_t * z_t
        sigma_t^2 = omega + alpha*epsilon_(t-1)^2
                   + gamma*I(epsilon_(t-1)<0)*epsilon_(t-1)^2
                   + beta*sigma_(t-1)^2
        z_t follows Hansen's standardized skewed Student-t distribution.

    Returns are in percentage points (e.g. 1% -> 1.0), not decimal units.
    """
    y = pd.Series(returns_pct, dtype=float).replace([np.inf, -np.inf], np.nan).dropna()
    if len(y) < MIN_TRAIN_OBSERVATIONS:
        raise ValueError(
            f"Only {len(y)} usable return observations; need at least "
            f"{MIN_TRAIN_OBSERVATIONS}. Reduce MIN_TRAIN_OBSERVATIONS only if justified."
        )

    model = arch_model(
        y,
        mean="Constant",
        vol="GARCH",
        p=1,
        o=1,           # o=1 adds the GJR/TARCH asymmetric news term
        q=1,
        power=2.0,
        dist="skewt",  # Hansen skewed Student-t innovations
        rescale=False,
    )
    result = model.fit(
        disp="off",
        show_warning=False,
        options={"maxiter": MAX_OPTIMIZER_ITERATIONS},
    )

    convergence_flag = int(getattr(result, "convergence_flag", 0))
    if convergence_flag != 0:
        raise RuntimeError(
            f"GJR-GARCH optimizer did not converge (convergence_flag={convergence_flag}). "
            "Do not silently report this fit; inspect the sample and optimizer settings."
        )

    params = result.params
    required = ["mu", "omega", "alpha[1]", "gamma[1]", "beta[1]", "eta", "lambda"]
    missing = [name for name in required if name not in params.index]
    if missing:
        raise KeyError(f"Unexpected arch parameter names; missing {missing}; found {list(params.index)}")

    residuals = pd.Series(result.resid).replace([np.inf, -np.inf], np.nan).dropna()
    conditional_vol = pd.Series(result.conditional_volatility).replace(
        [np.inf, -np.inf], np.nan
    ).dropna()
    if residuals.empty or conditional_vol.empty:
        raise RuntimeError("The fitted model did not return usable residuals/conditional volatility.")

    fitted = {name: float(params[name]) for name in required}
    fitted["last_residual"] = float(residuals.iloc[-1])
    fitted["last_sigma2"] = float(conditional_vol.iloc[-1] ** 2)
    fitted["convergence_flag"] = convergence_flag

    # For skewed innovations, E[z^2 I(z<0)] is not generally 1/2.
    # Use the fitted skew-t partial second moment for the GJR persistence diagnostic.
    skewt = SkewStudent()
    fitted["negative_partial_second_moment"] = float(
        skewt.partial_moment(2, z=0.0, parameters=[fitted["eta"], fitted["lambda"]])
    )
    fitted["persistence"] = (
        fitted["alpha[1]"]
        + fitted["gamma[1]"] * fitted["negative_partial_second_moment"]
        + fitted["beta[1]"]
    )
    fitted["n_fit_obs"] = int(len(y))
    return fitted


def one_step_variance(fit: Dict[str, float], previous_residual: float, previous_sigma2: float) -> float:
    """GJR one-step conditional variance, in squared percentage points."""
    eps2 = float(previous_residual) ** 2
    value = (
        fit["omega"]
        + fit["alpha[1]"] * eps2
        + fit["gamma[1]"] * eps2 * float(previous_residual < 0.0)
        + fit["beta[1]"] * float(previous_sigma2)
    )
    if not np.isfinite(value) or value <= 0.0:
        raise FloatingPointError(f"Invalid one-step conditional variance: {value}")
    return float(value)


def forecast_test_period(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate strictly one-step-ahead conditional density forecasts in test period."""
    data = df.copy().reset_index(drop=True)
    data["_return_pct"] = data["_log_return_decimal"] * 100.0
    data["_threshold_pct"] = data["event_threshold_decimal"] * 100.0

    test_positions = np.flatnonzero(data["date"].to_numpy() >= TEST_CUTOFF.to_datetime64())
    if len(test_positions) == 0:
        raise ValueError(f"No target observations on/after TEST_CUTOFF={TEST_CUTOFF.date()}.")

    forecasts: List[dict] = []
    fit_history: List[dict] = []
    fit_state: Optional[Dict[str, float]] = None
    last_successful_fit_end: Optional[pd.Timestamp] = None

    for test_count, target_pos in enumerate(test_positions):
        target_date = pd.Timestamp(data.loc[target_pos, "date"])

        # At target date t, only rows strictly before t are available.
        estimation_returns = data.loc[: target_pos - 1, "_return_pct"].dropna()
        if len(estimation_returns) < MIN_TRAIN_OBSERVATIONS:
            forecasts.append({
                "date": target_date,
                "actual": int(data.loc[target_pos, "crash_label"]),
                "actual_logreturn": data.loc[target_pos, "_log_return_decimal"],
                "event_threshold": data.loc[target_pos, "event_threshold_decimal"],
                "garch_probability": np.nan,
                "conditional_mean_pct": np.nan,
                "conditional_variance_pct2": np.nan,
                "conditional_volatility_pct": np.nan,
                "conditional_volatility_ann": np.nan,
                "realized_innovation_pct": np.nan,
                "fit_end_date": pd.NaT,
                "refit_performed": False,
                "fit_observations": len(estimation_returns),
            })
            continue

        do_refit = (fit_state is None) or (test_count % REFIT_EVERY == 0)
        if do_refit:
            try:
                fit_state = fit_gjr_skewt(estimation_returns)
                last_successful_fit_end = pd.Timestamp(
                    data.loc[: target_pos - 1, "date"].iloc[-1]
                )
                fit_history.append({
                    "forecast_date": target_date,
                    "fit_end_date": last_successful_fit_end,
                    "fit_observations": fit_state["n_fit_obs"],
                    "mu": fit_state["mu"],
                    "omega": fit_state["omega"],
                    "alpha": fit_state["alpha[1]"],
                    "gamma": fit_state["gamma[1]"],
                    "beta": fit_state["beta[1]"],
                    "eta": fit_state["eta"],
                    "lambda": fit_state["lambda"],
                    "persistence": fit_state["persistence"],
                    "convergence_flag": fit_state["convergence_flag"],
                })
            except Exception as exc:
                if fit_state is None:
                    raise RuntimeError(
                        f"Initial GJR-GARCH fit failed at {target_date.date()}: {exc}"
                    ) from exc
                warnings.warn(
                    f"GJR-GARCH refit failed at {target_date.date()}; retaining the last "
                    f"successful parameter/state estimate. Details: {exc}"
                )
                do_refit = False

        # Forecast sigma_t from the last known residual and conditional variance.
        sigma2_t = one_step_variance(
            fit_state,
            fit_state["last_residual"],
            fit_state["last_sigma2"],
        )
        sigma_t = float(np.sqrt(sigma2_t))
        mu_t = float(fit_state["mu"])

        threshold_pct = data.loc[target_pos, "_threshold_pct"]
        threshold_pct = float(threshold_pct) if pd.notna(threshold_pct) else np.nan

        if np.isfinite(threshold_pct):
            z_threshold = (threshold_pct - mu_t) / sigma_t
            skewt = SkewStudent()
            p_event = float(
                skewt.cdf(
                    np.asarray([z_threshold], dtype=float),
                    parameters=[fit_state["eta"], fit_state["lambda"]],
                )[0]
            )
            p_event = float(np.clip(p_event, 0.0, 1.0))
        else:
            p_event = np.nan

        actual_return_pct = data.loc[target_pos, "_return_pct"]
        actual_return_pct = float(actual_return_pct) if pd.notna(actual_return_pct) else np.nan
        actual_innovation_pct = actual_return_pct - mu_t if np.isfinite(actual_return_pct) else np.nan

        forecasts.append({
            "date": target_date,
            "actual": int(data.loc[target_pos, "crash_label"]),
            "actual_logreturn": data.loc[target_pos, "_log_return_decimal"],
            "event_threshold": data.loc[target_pos, "event_threshold_decimal"],
            "garch_probability": p_event,
            "conditional_mean_pct": mu_t,
            "conditional_variance_pct2": sigma2_t,
            "conditional_volatility_pct": sigma_t,
            "conditional_volatility_ann": sigma_t * np.sqrt(252.0) / 100.0,
            "realized_innovation_pct": actual_innovation_pct,
            "fit_end_date": last_successful_fit_end,
            "refit_performed": bool(do_refit),
            "fit_observations": fit_state["n_fit_obs"],
            "gjr_omega": fit_state["omega"],
            "gjr_alpha": fit_state["alpha[1]"],
            "gjr_gamma": fit_state["gamma[1]"],
            "gjr_beta": fit_state["beta[1]"],
            "skewt_eta": fit_state["eta"],
            "skewt_lambda": fit_state["lambda"],
            "gjr_persistence": fit_state["persistence"],
        })

        # Once the target-day close is observed, update the filter state for t+1.
        # This observation can affect only future forecasts, never the forecast for t.
        if np.isfinite(actual_return_pct):
            fit_state["last_residual"] = actual_return_pct - mu_t
            fit_state["last_sigma2"] = sigma2_t

    forecast_df = pd.DataFrame(forecasts).sort_values("date").reset_index(drop=True)
    fit_history_df = pd.DataFrame(fit_history)
    return forecast_df, fit_history_df

In [ ]:
# =============================================================================
# 4. LABEL THRESHOLD POLICIES AND METRICS
# =============================================================================


def sequential_f1_thresholding(probabilities, labels, initial_threshold=0.5, dates=None):
    from finder_core.public_outputs import sequential_thresholds
    if dates is None:
        raise ValueError('Supply benchmark label dates for outcome availability checks')
    calendar = pd.read_csv(DATA_DIR / 'OHLCV' / f"{MARKET}_df_OHLCV.csv")
    calendar_dates = pd.to_datetime(calendar['Date']).sort_values().drop_duplicates()
    next_dates = pd.Series(calendar_dates.shift(-1).to_numpy(), index=calendar_dates)
    cutoffs = pd.DatetimeIndex(pd.to_datetime(dates))
    targets = next_dates.reindex(cutoffs)
    if targets.isna().any():
        raise ValueError('Next trading date unavailable')
    if initial_threshold != 0.5:
        raise ValueError('The common FINDER policy starts at 0.5')
    thresholds = sequential_thresholds(probabilities, labels, targets.to_numpy(), cutoffs.to_numpy())
    return thresholds, (np.asarray(probabilities) >= thresholds).astype(int)


def safe_auc(y: np.ndarray, p: np.ndarray) -> float:
    return float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan


def safe_ap(y: np.ndarray, p: np.ndarray) -> float:
    return float(average_precision_score(y, p)) if int(np.sum(y)) > 0 else np.nan


def calculate_metrics(y: np.ndarray, p: np.ndarray, dates) -> dict:
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    fixed_pred = (p >= FIXED_PROBABILITY_THRESHOLD).astype(int)
    seq_thresholds, seq_pred = sequential_f1_thresholding(
        p, y, initial_threshold=INITIAL_SEQUENTIAL_THRESHOLD, dates=dates
    )

    result = {
        "n_test": int(len(y)),
        "n_anomalies": int(np.sum(y)),
        "event_prevalence": float(np.mean(y)),
        "ROC_AUC": safe_auc(y, p),
        "AP": safe_ap(y, p),
        "Brier": float(brier_score_loss(y, p)),
        "LogLoss": float(log_loss(y, np.clip(p, 1e-8, 1 - 1e-8), labels=[0, 1])),
        "QLIKE_innovation": np.nan,
        "fixed_threshold": float(FIXED_PROBABILITY_THRESHOLD),
        "fixed_precision": float(precision_score(y, fixed_pred, zero_division=0)),
        "fixed_recall": float(recall_score(y, fixed_pred, zero_division=0)),
        "fixed_f1": float(f1_score(y, fixed_pred, zero_division=0)),
        "fixed_mcc": float(matthews_corrcoef(y, fixed_pred)),
        "sequential_precision": float(precision_score(y, seq_pred, zero_division=0)),
        "sequential_recall": float(recall_score(y, seq_pred, zero_division=0)),
        "sequential_f1": float(f1_score(y, seq_pred, zero_division=0)),
        "sequential_mcc": float(matthews_corrcoef(y, seq_pred)),
        "sequential_final_threshold": float(seq_thresholds[-1]) if len(seq_thresholds) else np.nan,
    }
    return result


def moving_block_indices(
    n: int, block_size: int, rng: np.random.Generator
) -> np.ndarray:
    """Moving-block bootstrap indices preserving local ordering within blocks."""
    if n <= 0:
        raise ValueError("Cannot bootstrap an empty sample.")
    block_size = min(max(1, int(block_size)), n)
    starts = np.arange(0, n - block_size + 1)
    n_blocks = int(np.ceil(n / block_size))
    selected = rng.choice(starts, size=n_blocks, replace=True)
    return np.concatenate([np.arange(s, s + block_size) for s in selected])[:n]


def moving_block_confidence_intervals(
    y: np.ndarray,
    p: np.ndarray,
    block_size: int = BOOTSTRAP_BLOCK_SIZE,
    n_bootstrap: int = BOOTSTRAP_REPLICATES,
    random_state: int = BOOTSTRAP_RANDOM_STATE,
) -> dict:
    """95% moving-block bootstrap CIs for probability and fixed-threshold metrics."""
    y = np.asarray(y, dtype=int)
    p = np.asarray(p, dtype=float)
    rng = np.random.default_rng(random_state)
    metrics: Dict[str, List[float]] = {
        "ROC_AUC": [], "AP": [], "Brier": [],
        "fixed_precision": [], "fixed_recall": [], "fixed_f1": [],
    }

    for _ in range(n_bootstrap):
        idx = moving_block_indices(len(y), block_size, rng)
        yb, pb = y[idx], p[idx]
        fixed_pred = (pb >= FIXED_PROBABILITY_THRESHOLD).astype(int)
        values = {
            "ROC_AUC": safe_auc(yb, pb),
            "AP": safe_ap(yb, pb),
            "Brier": float(brier_score_loss(yb, pb)),
            "fixed_precision": float(precision_score(yb, fixed_pred, zero_division=0)),
            "fixed_recall": float(recall_score(yb, fixed_pred, zero_division=0)),
            "fixed_f1": float(f1_score(yb, fixed_pred, zero_division=0)),
        }
        for name, value in values.items():
            if np.isfinite(value):
                metrics[name].append(float(value))

    ci = {"bootstrap_replicates_requested": int(n_bootstrap),
          "bootstrap_block_size": int(min(block_size, len(y)))}
    for name, values in metrics.items():
        if values:
            ci[f"{name}_LOW"] = float(np.quantile(values, 0.025))
            ci[f"{name}_HIGH"] = float(np.quantile(values, 0.975))
            ci[f"{name}_bootstrap_valid"] = int(len(values))
        else:
            ci[f"{name}_LOW"] = np.nan
            ci[f"{name}_HIGH"] = np.nan
            ci[f"{name}_bootstrap_valid"] = 0
    return ci


def calculate_volatility_diagnostics(forecasts: pd.DataFrame) -> dict:
    """QLIKE and variance-MSE diagnostics using squared one-day innovations as proxy."""
    valid = forecasts.dropna(
        subset=["conditional_variance_pct2", "realized_innovation_pct"]
    ).copy()
    if valid.empty:
        return {"QLIKE_innovation": np.nan, "variance_proxy_MSE": np.nan}
    variance = np.maximum(valid["conditional_variance_pct2"].to_numpy(float), 1e-12)
    innovation2 = np.square(valid["realized_innovation_pct"].to_numpy(float))
    qlike = np.log(variance) + innovation2 / variance
    return {
        "QLIKE_innovation": float(np.mean(qlike)),
        "variance_proxy_MSE": float(np.mean(np.square(innovation2 - variance))),
    }

In [ ]:
# =============================================================================
# 5. OPTIONAL PAIRED COMPARISON WITH EXISTING ML PREDICTIONS
# =============================================================================


def paired_block_bootstrap_comparison(
    garch_predictions: pd.DataFrame,
    comparator_file: Path,
    comparator_columns: Sequence[str],
) -> pd.DataFrame:
    """Paired date-block bootstrap for Delta AUC/AP (GARCH minus each ML model)."""
    if not comparator_file.exists():
        raise FileNotFoundError(f"Comparator predictions file not found: {comparator_file}")
    comparator = pd.read_csv(comparator_file)
    if "date" not in comparator.columns:
        raise KeyError("Comparator predictions CSV must contain a 'date' column.")
    if "actual" not in comparator.columns:
        raise KeyError("Comparator predictions CSV must contain an 'actual' column.")

    comparator["date"] = pd.to_datetime(comparator["date"], errors="coerce")
    base = garch_predictions[["date", "actual", "garch_probability"]].copy()
    base["date"] = pd.to_datetime(base["date"], errors="coerce")
    merged = base.merge(comparator, on="date", how="inner", suffixes=("_garch", "_other"))
    if merged.empty:
        raise ValueError("No common dates between GARCH and comparator predictions.")
    if not np.array_equal(
        merged["actual_garch"].to_numpy(dtype=int),
        merged["actual_other"].to_numpy(dtype=int),
    ):
        raise ValueError("Actual labels disagree between GARCH and comparator files.")

    y = merged["actual_garch"].to_numpy(dtype=int)
    p_garch = merged["garch_probability"].to_numpy(dtype=float)
    rng = np.random.default_rng(BOOTSTRAP_RANDOM_STATE)
    rows = []

    for col in comparator_columns:
        if col not in merged.columns:
            warnings.warn(f"Comparator column '{col}' not found; skipping it.")
            continue
        p_other = pd.to_numeric(merged[col], errors="coerce").to_numpy(dtype=float)
        valid = np.isfinite(y) & np.isfinite(p_garch) & np.isfinite(p_other)
        yv, pg, po = y[valid], p_garch[valid], p_other[valid]
        if len(yv) < 2:
            continue

        auc_diff_samples: List[float] = []
        ap_diff_samples: List[float] = []
        for _ in range(BOOTSTRAP_REPLICATES):
            idx = moving_block_indices(len(yv), BOOTSTRAP_BLOCK_SIZE, rng)
            yb, pgb, pob = yv[idx], pg[idx], po[idx]
            auc_g, auc_o = safe_auc(yb, pgb), safe_auc(yb, pob)
            ap_g, ap_o = safe_ap(yb, pgb), safe_ap(yb, pob)
            if np.isfinite(auc_g) and np.isfinite(auc_o):
                auc_diff_samples.append(float(auc_g - auc_o))
            if np.isfinite(ap_g) and np.isfinite(ap_o):
                ap_diff_samples.append(float(ap_g - ap_o))

        auc_g, auc_o = safe_auc(yv, pg), safe_auc(yv, po)
        ap_g, ap_o = safe_ap(yv, pg), safe_ap(yv, po)
        rows.append({
            "Comparator": col,
            "n_common_dates": int(len(yv)),
            "event_prevalence": float(np.mean(yv)),
            "GARCH_ROC_AUC": auc_g,
            "Comparator_ROC_AUC": auc_o,
            "Delta_ROC_AUC_GARCH_minus_comparator": auc_g - auc_o,
            "Delta_ROC_AUC_CI_LOW": float(np.quantile(auc_diff_samples, 0.025)) if auc_diff_samples else np.nan,
            "Delta_ROC_AUC_CI_HIGH": float(np.quantile(auc_diff_samples, 0.975)) if auc_diff_samples else np.nan,
            "GARCH_AP": ap_g,
            "Comparator_AP": ap_o,
            "Delta_AP_GARCH_minus_comparator": ap_g - ap_o,
            "Delta_AP_CI_LOW": float(np.quantile(ap_diff_samples, 0.025)) if ap_diff_samples else np.nan,
            "Delta_AP_CI_HIGH": float(np.quantile(ap_diff_samples, 0.975)) if ap_diff_samples else np.nan,
            "bootstrap_block_size": min(BOOTSTRAP_BLOCK_SIZE, len(yv)),
            "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        })
    return pd.DataFrame(rows)

In [ ]:
# =============================================================================
# 6. MAIN
# =============================================================================


def main() -> None:
    np.random.seed(RANDOM_SEED)
    merged, return_source, threshold_source = load_and_align_data()

    print("\nFORECAST PROTOCOL")
    print("=" * 72)
    print(f"Test cutoff:        {TEST_CUTOFF.date()}")
    print("Forecast target:    same-day close-to-close log return r_t")
    print("Information cutoff: previous available trading close (t-1)")
    print(f"Initial estimation: expanding history, minimum {MIN_TRAIN_OBSERVATIONS} returns")
    print(f"Refit schedule:     every {REFIT_EVERY} target observations")
    print("Volatility model:   GJR-GARCH(1,1) with Hansen skewed Student-t errors")
    print(f"Return field:       {return_source}")
    print(f"Event threshold:    {threshold_source}")

    forecasts, fit_history = forecast_test_period(merged)
    valid = forecasts.dropna(subset=["garch_probability", "actual"]).copy()
    if valid.empty:
        raise ValueError(
            "No valid test forecasts were produced. Check the test cutoff, "
            "threshold availability, and minimum estimation sample size."
        )

    y = valid["actual"].to_numpy(dtype=int)
    p = valid["garch_probability"].to_numpy(dtype=float)

    # Hard labels under two policies: a common fixed threshold and the notebook's
    # sequential F1 policy. Ranking metrics (ROC-AUC/AP) are independent of thresholds.
    fixed_prediction = (p >= FIXED_PROBABILITY_THRESHOLD).astype(int)
    sequential_thresholds, sequential_prediction = sequential_f1_thresholding(
        p, y, initial_threshold=INITIAL_SEQUENTIAL_THRESHOLD, dates=valid["date"]
    )
    valid["fixed_threshold"] = FIXED_PROBABILITY_THRESHOLD
    valid["fixed_prediction"] = fixed_prediction
    valid["sequential_threshold"] = sequential_thresholds
    valid["sequential_prediction"] = sequential_prediction

    metrics = calculate_metrics(y, p, valid["date"])
    metrics.update(calculate_volatility_diagnostics(valid))
    ci = moving_block_confidence_intervals(y, p)
    metrics.update(ci)
    metrics_df = pd.DataFrame([{"Market": MARKET, "Model": "GJR-GARCH(1,1)-skewt", **metrics}])

    print("\nTEST PERFORMANCE")
    print("=" * 72)
    print(metrics_df.T.to_string(header=False))

    print("\nREFIT HISTORY")
    print("=" * 72)
    if not fit_history.empty:
        print(fit_history.to_string(index=False))
    else:
        print("No successful refits recorded.")

    if SAVE_RESULTS:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        forecast_path = OUTPUT_DIR / f"{MARKET}_GJR_GARCH_test_predictions.csv"
        metrics_path = OUTPUT_DIR / f"{MARKET}_GJR_GARCH_test_metrics.csv"
        fit_path = OUTPUT_DIR / f"{MARKET}_GJR_GARCH_fit_history.csv"
        valid.to_csv(forecast_path, index=False)
        metrics_df.to_csv(metrics_path, index=False)
        fit_history.to_csv(fit_path, index=False)
        print("\nSAVED OUTPUTS")
        print(f"Predictions: {forecast_path.resolve()}")
        print(f"Metrics:     {metrics_path.resolve()}")
        print(f"Fit history: {fit_path.resolve()}")

        if COMPARATOR_PREDICTIONS_CSV is not None:
            paired = paired_block_bootstrap_comparison(
                valid,
                Path(COMPARATOR_PREDICTIONS_CSV),
                COMPARATOR_PROBABILITY_COLUMNS,
            )
            paired_path = OUTPUT_DIR / f"{MARKET}_GJR_GARCH_paired_comparisons.csv"
            paired.to_csv(paired_path, index=False)
            print(f"Paired CIs:  {paired_path.resolve()}")
            if not paired.empty:
                print("\nPAIRED AUC/AP DIFFERENCES (GARCH - comparator)")
                print(paired.to_string(index=False))

    # Optional diagnostic plot. The CSV remains the authoritative reproducible output.
    try:
        import matplotlib.pyplot as plt

        fig, ax1 = plt.subplots(figsize=(14, 5))
        ax1.plot(valid["date"], valid["garch_probability"], label="GJR-skewt event probability")
        ax1.scatter(
            valid.loc[valid["actual"] == 1, "date"],
            valid.loc[valid["actual"] == 1, "garch_probability"],
            marker="x",
            label="Actual anomaly label = 1",
        )
        ax1.axhline(FIXED_PROBABILITY_THRESHOLD, linestyle="--", label="Fixed threshold")
        ax1.set_ylabel("P(return <= historical event threshold)")
        ax1.set_xlabel("Target date")
        ax1.set_title(f"{MARKET}: one-step-ahead GJR-GARCH(1,1)-skewt anomaly probabilities")
        ax1.grid(True, alpha=0.25)
        ax1.legend(loc="best")
        fig.tight_layout()
        if SAVE_RESULTS:
            fig.savefig(OUTPUT_DIR / f"{MARKET}_GJR_GARCH_test_probabilities.png", dpi=200)
        plt.show()

        fig, ax = plt.subplots(figsize=(14, 4))
        ax.plot(valid["date"], valid["conditional_volatility_ann"] * 100.0,
                label="Forecast annualized conditional volatility")
        ax.set_ylabel("Annualized volatility (%)")
        ax.set_xlabel("Target date")
        ax.set_title(f"{MARKET}: one-step-ahead GJR-GARCH conditional volatility")
        ax.grid(True, alpha=0.25)
        ax.legend(loc="best")
        fig.tight_layout()
        if SAVE_RESULTS:
            fig.savefig(OUTPUT_DIR / f"{MARKET}_GJR_GARCH_test_volatility.png", dpi=200)
        plt.show()
    except ImportError:
        print("matplotlib not installed; skipping plots (CSV outputs were saved).")


if __name__ == "__main__":
    main()
